# 01 · Forecasting and trading

**Order-Book Forecasting: Price, Liquidity, and Trading Value**

Can first-half order-book information (seconds 0–299) forecast the next-half mid-price return and
the change in average relative spread, and do those forecasts improve simulated trading decisions
after costs? Derived from the team's merged notebook (`archive/Optiver_Team_Merged.ipynb`); section
numbers are kept. All code lives in `src/obforecast/`, so the notebook runs top to bottom with no
hidden state.

The same analysis runs from the command line, writing every table to `results/metrics/` and every
figure to `results/figures/`:

```bash
python -m obforecast.run --config configs/default.yaml --data-dir <kaggle folder>          # development
python -m obforecast.run --config configs/default.yaml --data-dir <kaggle folder> --score-benchmark
```

> Committed **without outputs** (no data access when this version was prepared). Headline numbers
> quoted from the archived notebook are listed with their verification status in
> `results/metrics/headline_claims.csv`; they are deliberately not repeated as results here.

| § | Contents |
|---|---|
| 0 | Setup |
| 1 | Question, timing and hypotheses |
| 2 | Data and book reconstruction |
| 3 | Validation design and model inputs |
| 5 | Forecasts (H1, H2); 5.2 upgraded models; 5.3 drivers and unseen stocks |
| 6 | Trading simulation (H3, H4); 6.1 sensitivity |
| 7 | Why the spread forecast does (not) help; Figure 1 |
| 8 | Robustness: ablation, stress tests, random-split leakage, correctness checks |
| 9 | Legacy benchmark, scored once |
| 10 | Hypothesis summary from computed results; limitations |

## 0. Setup

Runs locally or in Colab. Data comes from the Kaggle competition
[`optiver-realized-volatility-prediction`](https://www.kaggle.com/competitions/optiver-realized-volatility-prediction/data)
(accept the competition rules on Kaggle first). Set `DATA_DIR` below, or the `OBF_DATA_DIR`
environment variable, to the extracted folder that contains `train.csv`, `book_train.parquet/` and
`trade_train.parquet/`. In Colab, the next cell clones the repository, installs the requirements and,
if `DATA_DIR` is empty, downloads the data with a `KAGGLE_API_TOKEN` stored in Colab Secrets.

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Erdos-Projects/fall-2026-short-horizon-forecasting"
DATA_DIR = os.environ.get("OBF_DATA_DIR", "")      # e.g. "/path/to/optiver-realized-volatility-prediction"
CONFIG = "configs/smoke.yaml"                       # "configs/default.yaml" for the full analysis

if IN_COLAB:
    if not Path("fall-2026-short-horizon-forecasting").exists():
        !git clone -q {REPO_URL}
    %cd fall-2026-short-horizon-forecasting
    %pip install -q -r requirements.txt
    %pip install -q -e .
    if not DATA_DIR:
        from google.colab import userdata
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
        DATA_DIR = "/content/data/optiver-realized-volatility-prediction"
        kagglehub.competition_download("optiver-realized-volatility-prediction", output_dir=DATA_DIR)
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(root)
    sys.path.insert(0, str(root / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from obforecast.config import Timing, load_config
from obforecast.data import ParquetSource, check_layout, resolve_data_dir, select_stocks
from obforecast.book import build_tables
from obforecast import pipeline

cfg = load_config(CONFIG)
timing = Timing.from_config(cfg)
data_dir = resolve_data_dir(DATA_DIR or None, cfg)
layout = check_layout(data_dir)
source = ParquetSource(data_dir)
stock_ids = select_stocks(layout["book_stocks"], cfg["run"]["stocks"])
print(f"run label: {cfg['run']['label']} | data: {data_dir} | stocks used: {len(stock_ids)} of {layout['train_stocks']}")
print({k: v for k, v in layout.items() if k not in ("book_stocks",)})

SCORE_BENCHMARK = False       # set True only for the single, final benchmark run (section 9)
RUN_UPGRADED = True           # sections 5.2/5.3/9.1 need lightgbm and are slow on the full data

## 1. Question, timing and hypotheses

Each sample is one stock in one 10-minute window. Features use seconds 0–299; the decision is taken
right after second 299; the simulated trade buys one share at the ask at second 300 (robustness:
302, 305, 310) and sells at the bid at second 599.

| | Hypothesis | Tested in |
|---|---|---|
| H1 | The spread change is predictable beyond stock averages and persistence. | §5, §9 |
| H2 | The price change is predictable, mostly through the market (other stocks). | §5, §7, §8 |
| H3 | A price forecast clears the cost of crossing the spread often enough to earn a positive net return. | §6, §8, §9 |
| H4 | Using the spread forecast as a filter improves the tested trading rule. | §6, §7 |

Any of these can come out negative. This is a simulation; nothing connects to a broker.

## 2. Data and book reconstruction

`build_tables` returns three tables keyed by `(stock_id, time_id)`: **features** (seconds 0–299:
spread level and trend, minimum-spread proxies, WAP volatility, recent returns and book tilt, quote
activity and age, depth and imbalance), **outcomes** (the two targets and the quotes the simulator
needs; never model inputs) and **quality** rows.

In [ ]:
# One pass over each stock's book: features (seconds 0..299), outcomes, data quality.
# Cached by obforecast.run under cache/ when you use the command line; here we build directly.
feats, outs, quality, invalid = build_tables(source, stock_ids, timing)
d = pipeline.prepare(feats, outs, cfg)   # legacy benchmark split + outer grouped folds
print(d.split_info)

## 3. Validation design and model inputs

1. **Group by window.** All stocks of a `time_id` share a fold (stocks in a window share a market move).
2. **Nest every choice.** Heads and trading rules are picked on inner folds and scored once on an outer
   fold they never saw.
3. **Legacy benchmark.** 766 windows (seed-0 recipe; identifiers saved to `results/splits/`) are set
   aside until §9. These windows were **previously examined** by earlier team experiments, including
   price models on a different split (`archive/README.md`), so they are a consistency check, not an
   untouched test.

Model inputs (47): 30 first-half features in 7 families, 4 stock-relative versions (training-fold
stock means), 11 leave-one-out market means and 2 relative-to-market inputs. The upgraded models of
§5.2 use 131 inputs; the full list is in `results/metrics/feature_inventory.csv`.

In [ ]:
from obforecast.features import FEATURES, family, feature_inventory
inv = feature_inventory()
print(f"baseline inputs: {inv.baseline_47.sum()} | upgraded inputs: {inv.upgraded_131.sum()}")
display(pd.Series([family(c) for c in FEATURES]).value_counts().rename("baseline inputs per family").to_frame())

## 5. Forecasts: baselines and two simple models (H1, H2)

Baselines: zero change, the stock's training-fold mean, and persistence (price: zero; spread: the
last-60 s mean spread persists). Heads: median imputation + scaling + ridge (α from the config) and one
fixed `HistGradientBoostingRegressor`. In each outer fold the head with the lowest inner out-of-fold
RMSE is selected per target, refit on the outer training fold and scored on the outer validation fold.
The trading rules of §6 are chosen in the same loop.

In [ ]:
cv = pipeline.nested_cv(d, cfg, timing)
display(cv.selections)

### 5.1 Forecast accuracy (outer folds)

RMSE is in each target's own units; do not compare it across targets. Historical reference values
from the archived notebook's saved output are spread R² ≈ 0.3484 and price R² ≈ 0.0612 (development,
HGB and ridge selected). They are references, not targets.

In [ ]:
acc = pd.concat([pipeline.forecast_table(t, d, cv.oof) for t in ("price", "spread")])
display(acc.set_index(["target", "model"]).round(4))
cal = pd.concat([pipeline.calibration(t, d, cv.oof) for t in ("price", "spread")])
from obforecast import figures
figures.calibration_figure(cal, Path(cfg["run"]["output_dir"]) / "figures" / "calibration.png")
display(cal.round(3))

### 5.2 Upgraded models

Same folds, everything fitted on training folds: the 31 extra first-half inputs (spread path, depth
trend, trade prints), more market context, stock target statistics and a categorical stock id; the
spread uses LightGBM averaged over seeds and the price a 50/50 blend of a Huber LightGBM and a ridge.
**The trading results in §6–§9 use the §5 models, not these.**

In [ ]:
from obforecast.extra import extra_stock
if RUN_UPGRADED:
    extra = pd.concat([extra_stock(s, source.load("book", s), source.load("trade", s), timing) for s in stock_ids],
                      ignore_index=True)
    up = pipeline.upgraded_cv(d, extra, cv, cfg)
    display(up["table"].round(4))

### 5.3 Drivers and unseen stocks

(a) Mean absolute LightGBM contribution per input (models fitted on folds 2–5, explained on a sample
of fold 1). (b) Spread skill on stocks **and** windows never seen in training (stock group k ×
window fold k), compared on the same rows with a model that has seen those stocks.

In [ ]:
if RUN_UPGRADED:
    chk = pipeline.upgraded_checks(d, up["dev_x"], cfg)
    top = chk["drivers"].sort_values("mean_abs_contribution", ascending=False).groupby("target").head(12)
    display(top.round(5))
    u = chk["unseen"].pivot(index="fold", columns="setting", values="spread R2")
    u.loc["mean"] = u.mean()
    display(u.round(3))

## 6. Trading simulation (H3, H4)

| Policy | Enter when |
|---|---|
| Cash | never |
| Price only | forecast `y_price` > current relative spread + 2 × cost per side + margin |
| Price + spread | the same, and forecast `y_spread ≤ τ` |

Buy one share at the ask at 300 and sell at the bid at 599, with a proportional cost per side
(0, 0.5, 1 bp; 0.5 is the reference). Crossing the quotes already pays the spread, so it is not
subtracted again. A leg without a quote or size is *unresolved* and earns 0 rather than being
dropped. The main metric is **net bps per opportunity** (no trade = 0). Prices are normalised and
`time_id`s are shuffled, so there is no dollar P&L, equity curve, Sharpe ratio or drawdown.

In [ ]:
tr = pipeline.trading(d, cv, cfg, timing)
display(tr["summary"].set_index("policy").round(3))
display(tr["bootstrap"].round(4))
print(tr["window_t"])

### 6.1 Sensitivity

(1) Fixed decisions under each cost, later entry and stale-ask filters; (2) rules re-selected under
each cost from the same inner forecasts (a different question, so a separate table); (3) stock groups.

In [ ]:
sens = pipeline.sensitivity(d, cv, tr["ENTER"], cfg, timing)
cols = ["entries", "net bps per trade", "net bps per opportunity", "win rate"]
display(sens["fixed"].set_index(["policy", "check"])[cols].round(3))
display(sens["retuned"].set_index(["cost bp/side", "policy"])[cols].round(3))
display(sens["groups"].set_index(["policy", "group"])[["opportunities"] + cols].round(3))

## 7. Why the spread forecast does or does not help the tested rule

(a) A direct forecast of the realised crossing cost vs the current spread `q_299` used by the hurdle;
(b) an oracle rule given the exact realised cost (not tradable: if it does not help, no spread forecast
can help **this** rule); (c) a market-only price model (window-mean return forecast × stock beta); (d)
calibration of the price forecast by decile.

In [ ]:
why = pipeline.why_spread(d, cv, tr["ENTER"], cfg, timing)
display(why["diagnostics"].round(4)); display(why["policies"].round(4)); display(why["deciles"].round(2))

### Figure 1. Forecast accuracy vs trading value

In [ ]:
skill = pd.DataFrame({"model": ["zero", "stock mean", "persistence", "ridge", "HGB"]})
for t, col in [("spread", "y_spread"), ("price", "y_price")]:
    skill[t] = [pipeline.r2(d.dev_o[col].to_numpy(), cv.oof[f"{t}: {m}"].to_numpy()) for m in skill.model]
vals = []
for c in cfg["trading"]["cost_scenarios_bps"]:
    po = pipeline.per_opportunity(tr["ENTER"]["price only"], d.dev_o, c, timing)
    m, lo, hi = pipeline.paired_bootstrap(po, np.zeros(len(po)), d.dev_f.time_id.to_numpy(),
                                          cfg["bootstrap"]["resamples"], cfg["bootstrap"]["seed"])
    vals.append({"cost_bps": c, "mean_bps": m, "ci_low": lo, "ci_high": hi})
out = Path(cfg["run"]["output_dir"]) / "figures" / "figure1_accuracy_vs_value.png"
figures.figure1(skill, pd.DataFrame(vals), out, f"{cfg['run']['label']} run, development windows, nested grouped CV")
from IPython.display import Image
Image(str(out))

## 8. Robustness

### 8.1 Feature ablation
One family removed at a time with all of its stock-relative, market and relative versions; fast HGB
on 3 grouped folds, so compare rows within this table only.

In [ ]:
display(pipeline.ablation(d, cfg).round(4))

### 8.2 Stress tests
(a) When does the gain accrue, and do stale quotes earn more? (b) lagged-by-30 s and shuffled-across-
windows market inputs (placebo), own-stock inputs only; (c) concentration in the best windows, with a
frictionless always-long benchmark trimmed the same way; (d) neighbouring margins and break-even fee.
For speed (b) uses plain grouped CV with a fixed rule, so its "as built" row differs slightly from §6.

In [ ]:
from obforecast.extra import path_points
from obforecast.book import KEYS
pts = pd.concat([path_points(s, source.load("book", s), cfg["robustness"]["path_seconds"], timing) for s in stock_ids],
                ignore_index=True)
st = pipeline.stress(d, cv, tr["ENTER"], d.dev_f[KEYS].merge(pts, on=KEYS, how="left"), cfg, timing)
for k, v in st.items():
    print(k); display(v.round(3) if isinstance(v, pd.DataFrame) else v)

### 8.3 Leakage from a random split
The same HGB scored with folds grouped by window and with random rows. Stock means are learned on
each training fold (the archived notebook learned them once on all development rows).

In [ ]:
display(pipeline.leakage_demo(d, cfg).round(3))

### 8.4 Correctness checks
Key uniqueness and row alignment, group separation (outer and inner), future-data perturbation of
**baseline and upgraded book/trade inputs**, forward-fill boundaries, the flat round trip, cost
monotonicity, price-scale invariance, and the decision function's inputs. The same checks run on
synthetic books in `tests/`.

In [ ]:
from obforecast.checks import run_checks
sid = stock_ids[0]
checks = run_checks(feats, outs, d.dev_f, cfg, timing,
                    sample_stock=(sid, source.load("book", sid), source.load("trade", sid)))
display(checks)
assert checks.passed.all(), list(checks.check[~checks.passed])

## 9. Legacy benchmark, scored once

The pipeline is frozen (heads and both policies chosen on inner folds of all development rows, refit
on all development rows) and scored once on the benchmark windows. Unlike the archived notebook, both
policies are selected and scored, so the spread filter is actually tested here. Remember that these
windows were previously examined (§3).

In [ ]:
if SCORE_BENCHMARK:
    bench = pipeline.benchmark_run(d, cfg, timing)
    display(bench["skill"].round(4)); display(bench["trading"].round(3)); print(bench["frozen"])
    if RUN_UPGRADED:
        display(pipeline.upgraded_benchmark(d, extra, bench, cfg).round(4))
else:
    print("Benchmark not scored (SCORE_BENCHMARK = False).")

## 10. Hypotheses, from the computed results

The verdict table below is computed from this run, not written by hand.

In [ ]:
s = tr["summary"].set_index("policy"); b = tr["bootstrap"].set_index("comparison")
acc_i = acc.set_index(["target", "model"])
tau_chosen = cv.selections["price + spread: tau"].replace(np.inf, np.nan).notna().sum()
verdicts = pd.DataFrame([
    ("H1 spread predictable", f"selected R2 {acc_i.loc[('spread', 'selected'), 'R2']:.3f} vs stock mean "
     f"{acc_i.loc[('spread', 'stock mean'), 'R2']:.3f} and persistence {acc_i.loc[('spread', 'persistence'), 'R2']:.3f}"),
    ("H2 price predictable", f"selected R2 {acc_i.loc[('price', 'selected'), 'R2']:.3f}; market-only share "
     f"{why['diagnostics'].set_index('quantity').loc['market-only share of price R2', 'value']:.0%}"),
    ("H3 net return after costs", f"price only minus cash {b.loc['price only minus cash', 'mean_bps']:+.3f} bps/opp "
     f"(95% CI {b.loc['price only minus cash', 'ci_low']:+.3f} to {b.loc['price only minus cash', 'ci_high']:+.3f})"),
    ("H4 spread filter improves the tested rule", f"filter chosen in {tau_chosen} of {len(cv.selections)} folds; "
     f"difference {b.loc['price + spread minus price only', 'mean_bps']:+.4f} bps/opp "
     f"(CI {b.loc['price + spread minus price only', 'ci_low']:+.4f} to {b.loc['price + spread minus price only', 'ci_high']:+.4f})"),
], columns=["hypothesis", "evidence (this run)"])
pd.set_option("display.max_colwidth", 200); display(verdicts)

**Limits that apply whatever the numbers say**
* **No time order.** `time_id`s are shuffled, so the same days can appear in every fold and in the
  benchmark; this is not an out-of-time test.
* **Previously examined benchmark.** See §3 and `archive/README.md`.
* **Simplified execution.** One share at the displayed quote; no queue position, market impact,
  latency beyond 10 s or real fee schedule; normalised prices, so no dollar capacity.
* **Concentration.** The effective sample is windows, not trades; check §8.2 for how much of the
  simulated P&L sits in a few windows.
* **Scope of H4.** A negative H4 means the tested spread filter did not improve the tested
  price-hurdle rule; it does not show that spread forecasts cannot improve trading in general.